# Tutorial 3. Bayesian statistics in search for energy extrema

## Goal of the notebook

The goal of this notebook is provide step-by-step instructions on how to use the BOSS method to find the adsoprtion site of OH intermediate on a PtN₄-carbon single-atom catalyst.

## Description of the method

Bayesian Optimisation Structure Search (BOSS) is a method for exploring atomistic configuration spaces and identifying stable structures using a small number of energy evaluations. The method combines Bayesian optimisation with DFT calculations to model the potential energy surface during the search. A surrogate model, usually a Gaussian process, is updated with energies computed for selected atomic configurations. New configurations are selected to sample unexplored regions of the configuration space and configurations with lowest energy.

BOSS is used for problems where each energy evaluation is expensive, such as density functional theory calculations of adsorption structures or molecular configurations on surfaces. During the search the model learns the energy landscape and identifies global and local minima without grid sampling or manual selection of structures.

BOSS webpage: https://cest-group.gitlab.io/boss/tutorials.html

BOSS github: https://gitlab.com/cest-group/boss

BOSS article: Todorović, M.; Gutmann, M. U.; Corander, J.; Rinke, P. Bayesian Inference of Atomistic Structure in Functional Materials. *npj Computational Materials* **2019**, *5*, 35. [https://doi.org/10.1038/s41524-019-0175-2](https://doi.org/10.1038/s41524-019-0175-2).

## Outline of the tutorial

STEP 1. Creating the surface + adsorbate

STEP 2. Finding the asdorption site for OH

The tutorial takes less than 2 minutes to run on 1 CPU in Google Colab.

## Installation of packages

First thing first, we need to install and import the packages.

Next we import the necessary packages:

1. ASE https://ase-lib.org/
2. BOSS https://cest-group.gitlab.io/boss
3. TBLite https://tblite.readthedocs.io

In [2]:
##############
##-Imports-###
##############

##-Imports for ASE
from ase import Atoms, Atom
from ase.build import fcc111, graphene
from ase.io import read, write
from ase.visualize import view

##-Imports for Boss
from boss.bo.bo_main import BOMain

##-Import for TBLite
from tblite.ase import TBLite

##-General imports
import numpy as np

/usr/local/lib/python3.12/dist-packages/ase/neighborlist.py:6: UserWarning: A NumPy version >=1.22.4 and <1.29.0 is required for this version of SciPy (detected version 2.0.2)
  import scipy.sparse.csgraph as csgraph


ModuleNotFoundError: No module named 'numpy.strings'

## STEP 1. Creating the surface + adsorbate

At the first step, we create the surface using ASE functions and run single point calculations on H₂O, H₂ and OH.

In [ ]:
##########################
##-Creating the surface-##
##########################

# Start with a graphene sheet
PtN4 = graphene(size=(4,4,1))
# Substitute 4 C atoms with N, add Pt and remove 2 C atoms
for i in (0, 3, 8, 11):
    PtN4[i].symbol = "N"
PtN4.append(Atom('Pt', position=(PtN4[1].position+PtN4[10].position)/2))
del PtN4[[10, 1]]
# Change the unit cell, shift Pt to the center
PtN4.cell[1,0] *= -1
PtN4.center(vacuum=5, axis=2)
shift = PtN4.cell[:2,:2].sum(axis=0) / 2 - PtN4[-1].position[:2]
PtN4.translate([shift[0], shift[1], 0.0])
PtN4.wrap()
PtN4.pbc = (True, True, False)

In [ ]:
####################
##-Defining species-##
######################

##-This sets up species needed for calculations
OH       = Atoms('OH', positions=[[0,0,0], [0,0,0.96]])
water    = Atoms('H2O',positions=[[0.75,0.58,0],[-0.75,0.58,0],[0,0,0]])
hydrogen = Atoms('H2', positions=[[0,0,0],[0.7,0,0]])

In [ ]:
###############################
##-Single point calculations-##
###############################

PtN4.calc     = TBLite(method="GFN1-xTB")
water.calc    = TBLite(method="GFN1-xTB")
hydrogen.calc = TBLite(method="GFN1-xTB")

slab_energy  = PtN4.get_potential_energy()
water_energy = water.get_potential_energy()
hydro_energy = hydrogen.get_potential_energy()

## STEP 2. Finding the preferred asdorption site for OH

At the second step, we define two functions. The first function places our adsorbate on 6 coordinates. The second function represents BOSS taking the coordinates and returning a single scalar. Now, using the defined functions and the BOSS object, we run the calculations to get the predicted global energy minimum.

In [ ]:
########################
##-Defining functions-##
########################

##-This function will place our adsorbate on given 6 coordinates
def build_system(x,y,z,rx,ry,rz, slab: Atoms, adsorbate: Atoms,) -> Atoms:
    slab = slab.copy()
    adsorbate = adsorbate.copy()
    adsorbate.rotate(rx,'x')
    adsorbate.rotate(ry,'y')
    adsorbate.rotate(rz,'z')
    adsorbate.translate((x,y,z))
    full_system = slab + adsorbate
    return full_system

##-This function represents one step of BOSS taking the 6 dimensional coordinates and returns a single scalar
def calculate_system(x,y,z,rx,ry,rz, slab: Atoms, adsorbate: Atoms, slab_energy:float, adsorbate_energy: float | list[float]) -> float:
    full_system = build_system(x,y,z,rx,ry,rz, slab, adsorbate)
    full_system.calc = TBLite(method="GFN1-xTB", verbosity = 0)
    full_system.calc.txt = f'full_system_{x}-{y}-{z}_{rx}-{ry}-{rz}.txt'
    if isinstance(adsorbate_energy, list) or isinstance(adsorbate_energy, tuple): adsorbate_energy = sum(adsorbate_energy)
    return full_system.get_potential_energy() - slab_energy - adsorbate_energy

In [ ]:
%%time

##########################
##-Optimizing structure-##
##########################

BO_obj = BOMain(
    f= lambda args: calculate_system(args[0,0],args[0,1],args[0,2],args[0,3],args[0,4],args[0,5],
                                     PtN4, OH,
                                     slab_energy=slab_energy,
                                     adsorbate_energy=(water_energy, -0.5*hydro_energy)),
    bounds=np.array([[3/7*(PtN4.cell[0,0] + PtN4.cell[1,0]), 4/7*(PtN4.cell[0,0] + PtN4.cell[1,0])],
                    [ 2/5*PtN4.cell[1,1], 3/5*PtN4.cell[1,1]],
                    [2/3*PtN4.cell[2,2], 3/4*PtN4.cell[2,2]],
                    [-30, 30],
                    [-30, 30],
                    [  0, 60],
                     ]
                    ),
    kernel='rbf',
    initpts=5,
    iterpts=16
)

results = BO_obj.run()

print('Predicted global min: ', results.select('mu_glmin', -1))
print(results.select('x_glmin', -1))

In [ ]:
PtN4_opt = build_system(*results.select('x_glmin', -1), slab = PtN4, adsorbate = OH)
PtN4_opt.wrap()
view(PtN4_opt, viewer='x3d')

## Concluding remarks

In this tutorial we applied BOSS to study OH adsorption on a PtN₄–carbon single‑site catalyst. We searched for the configuration with the lowest adsorption energy. This search identified that OH prefers to adsorb ontop Pt atom. This is the adsorption site a chemist would expect to be most reactive.

This example shows how BOSS can screen thought adsorption configurations to learning the energy surface and reveal the lowest energy configurations.

After completing this tutorial we can:

* Set up and run a BOSS search for an adsorption problem.

* Define the degrees of freedom that describe the adsorbate configuration.

* Build and update a Gaussian process model of the adsorption energy surface during the search.

* Use the model to guide the search toward the lowest‑energy adsorption configuration.